# 06 — Demo tata kelola TI (Bab 3)

Bab 3 sebagian besar berupa **rancangan**. Notebook ini membuktikan bagian yang **bisa dijalankan nyata** di HDFS kita:

| Demo | Bab | Bukti |
|---|---|---|
| 1. Hak akses per zona & per peran (permission + ACL) | 3.1 Security | tabel uji: siapa boleh apa, hasil nyata vs yang diharapkan |
| 2. Kelemahan autentikasi `simple` (identitas bisa dipalsukan) | 3.1 Security | perintah berhasil dengan identitas palsu → alasan Kerberos |
| 3. Audit log NameNode (siapa, kapan, apa, path, diizinkan?) | 3.1 / 3.2 | potongan `hdfs-audit.log` |
| 4. Lineage (asal-usul setiap file) | 3.2 Compliance | `lineage.jsonl` |
| 5. Rancangan yang tidak diimplementasikan | 3.1–3.4 | tabel |

**Peran (role):**
| Peran | User | Hak |
|---|---|---|
| Data engineer | `engineer` (group `engineer`) | baca raw, **tulis** processed & quality |
| Data scientist | `scientist` (group `scientist`) | baca raw, baca processed & quality, **tidak bisa menulis apa pun** |
| Pihak luar | `guest` (tanpa group) | hanya baca raw (dataset publik), tidak bisa melihat processed |
| Admin / superuser | `axelnakata` (yang menjalankan NameNode) | semua. Superuser HDFS **melewati** pemeriksaan permission |

**Konfigurasi yang ditambahkan** (satu kali, lalu HDFS di-restart; data tidak tersentuh):
- `core-site.xml` → `hadoop.user.group.static.mapping.overrides = dr.who=;engineer=engineer;scientist=scientist` (memetakan user → group di Hadoop tanpa membuat user macOS / tanpa `sudo`)
- `hadoop-env.sh` → `export HDFS_AUDIT_LOGGER=INFO,RFAAUDIT` (audit log ditulis ke `~/opt/hadoop/logs/hdfs-audit.log`; default-nya dibuang ke `NullAppender`)

In [1]:
import sys, os, re, json, subprocess
sys.path.insert(0, os.path.abspath('../src'))
from common import *
import pandas as pd
pd.set_option('display.max_colwidth', 120, 'display.width', 220)
sh('hdfs getconf -confKey hadoop.user.group.static.mapping.overrides')
sh('hdfs getconf -confKey hadoop.security.authentication')
for u in ('engineer', 'scientist', 'guest'):
    sh(f'hdfs groups {u}')

$ hdfs getconf -confKey hadoop.user.group.static.mapping.overrides
dr.who=;engineer=engineer;scientist=scientist


$ hdfs getconf -confKey hadoop.security.authentication
simple


$ hdfs groups engineer
engineer : engineer


$ hdfs groups scientist
scientist : scientist


$ hdfs groups guest
guest :


## 1. Pasang hak akses per zona

| Zona | Owner:group | Mode | ACL tambahan | Artinya |
|---|---|---|---|---|
| `/datalake/raw` | axelnakata:supergroup | `555` (r-x r-x r-x) | – | semua boleh baca, **tidak ada** yang boleh menulis/menghapus (kecuali superuser) |
| `/datalake/processed`, `/datalake/quality` | axelnakata:**engineer** | `770` (rwx rwx ---) | `group:scientist:r-x` (+ default ACL untuk file baru) | engineer tulis, scientist baca, orang lain tidak bisa apa-apa |

ACL (Access Control List) = izin tambahan di luar 3 slot owner/group/others. Tanpa ACL, 1 folder hanya bisa punya 1 group.

In [2]:
for z in (PROCESSED, QUALITY):
    sh(f'hdfs dfs -chgrp -R engineer {z}')
    sh(f'hdfs dfs -chmod -R ug=rwX,o-rwx {z}', quiet=True)
    sh(f'hdfs dfs -setfacl -R -m group:scientist:r-x {z}', quiet=True)
    dirs = [l.split()[-1] for l in sh(f'hdfs dfs -ls -R {z}', quiet=True).splitlines() if l.startswith('d')] + [z]
    for d in dirs:   # default ACL: file/folder baru (mis. hasil Spark berikutnya) otomatis ikut aturan yang sama
        sh(f'hdfs dfs -setfacl -m default:group:engineer:rwx,default:group:scientist:r-x,default:other::--- {d}', quiet=True)
sh(f'hdfs dfs -ls /datalake {RAW} {PROCESSED}')
sh(f'hdfs dfs -getfacl {PROCESSED}')

$ hdfs dfs -chgrp -R engineer /datalake/processed



$ hdfs dfs -chgrp -R engineer /datalake/quality



$ hdfs dfs -ls /datalake /datalake/raw /datalake/processed
Found 4 items
drwxr-xr-x   - axelnakata supergroup          0 2026-09-23 22:37 /datalake/_lineage
drwxrwx---+  - axelnakata engineer            0 2026-09-23 22:52 /datalake/processed
drwxrwx---+  - axelnakata engineer            0 2026-09-23 23:04 /datalake/quality
drwxr-xr-x   - axelnakata supergroup          0 2026-09-22 22:18 /datalake/raw
Found 3 items
dr-xr-xr-x   - axelnakata supergroup          0 2026-09-23 01:58 /datalake/raw/comcat
dr-xr-xr-x   - axelnakata supergroup          0 2026-09-22 22:29 /datalake/raw/exotic
dr-xr-xr-x   - axelnakata supergroup          0 2026-09-22 23:18 /datalake/raw/noise
Found 2 items
drwxrwx---+  - axelnakata engineer            0 2026-09-23 22:45 /datalake/processed/metadata
drwxrwx---+  - axelnakata engineer            0 2026-09-23 23:04 /datalake/processed/waveforms


$ hdfs dfs -getfacl /datalake/processed
# file: /datalake/processed
# owner: axelnakata
# group: engineer
user::rwx
group::rwx
group:scientist:r-x
mask::rwx
other::---
default:user::rwx
default:group::rwx
default:group:engineer:rwx
default:group:scientist:r-x
default:mask::rwx
default:other::---


'# file: /datalake/processed\n# owner: axelnakata\n# group: engineer\nuser::rwx\ngroup::rwx\ngroup:scientist:r-x\nmask::rwx\nother::---\ndefault:user::rwx\ndefault:group::rwx\ndefault:group:engineer:rwx\ndefault:group:scientist:r-x\ndefault:mask::rwx\ndefault:other::---'

## 2. Uji: siapa boleh apa?
Setiap aksi dijalankan sebagai user yang berbeda (`HADOOP_USER_NAME=<user>`). Kolom `expected` = aturan yang kita rancang, `actual` = jawaban HDFS.

Catatan keamanan uji: untuk raw zone kita hanya menguji **membuat** file. Izin hapus di HDFS memakai izin tulis yang sama pada folder induk, jadi kalau membuat file ditolak, menghapus juga ditolak. Kita tidak mencoba menghapus data asli.

In [3]:
def run_as(user, cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, env={**os.environ, 'HADOOP_USER_NAME': user})
    out = (r.stdout + r.stderr).strip()
    return 'ALLOWED' if r.returncode == 0 else ('DENIED' if 'Permission denied' in out else 'ERROR: ' + out[-120:])

CSV = f'{RAW}/exotic/exotic_metadata.csv'
PQ = f'{PROCESSED}/metadata'
ACTIONS = [  # (aksi, perintah, {user: expected})
 ('baca raw (CSV)',               f'hdfs dfs -cat {CSV} | head -c 100 > /dev/null; hdfs dfs -test -r {CSV}', dict(engineer='ALLOWED', scientist='ALLOWED', guest='ALLOWED')),
 ('tulis file baru di raw',       'hdfs dfs -touchz {raw}/exotic/_acl_test_{u}',                           dict(engineer='DENIED',  scientist='DENIED',  guest='DENIED')),
 ('lihat isi processed',          f'hdfs dfs -ls {PQ}',                                                     dict(engineer='ALLOWED', scientist='ALLOWED', guest='DENIED')),
 ('baca file Parquet processed',  f'hdfs dfs -cat $(hdfs dfs -ls -C {PQ}/dataset=exotic | grep parquet | head -1) | head -c 4 > /dev/null; hdfs dfs -test -r $(hdfs dfs -ls -C {PQ}/dataset=exotic | grep parquet | head -1)',
                                                                                                            dict(engineer='ALLOWED', scientist='ALLOWED', guest='DENIED')),
 ('tulis file baru di processed', 'hdfs dfs -touchz {processed}/_acl_test_{u} && hdfs dfs -rm -skipTrash {processed}/_acl_test_{u}',
                                                                                                            dict(engineer='ALLOWED', scientist='DENIED',  guest='DENIED')),
 ('baca laporan quality',         f'hdfs dfs -cat {QUALITY}/metadata_quality.json > /dev/null',             dict(engineer='ALLOWED', scientist='ALLOWED', guest='DENIED')),
]
rows = []
for name, cmd, exp in ACTIONS:
    for u, e in exp.items():
        a = run_as(u, cmd.format(u=u, raw=RAW, processed=PROCESSED))
        rows.append({'action': name, 'user': u, 'expected': e, 'actual': a, 'ok': '✅' if a == e else '❌'})
acl = pd.DataFrame(rows)
display(acl.pivot(index='action', columns='user', values='actual').reindex([a[0] for a in ACTIONS]))
assert (acl.ok == '✅').all(), acl[acl.ok != '✅']
print('Semua hasil sesuai rancangan hak akses.')

user,engineer,guest,scientist
action,,,
baca raw (CSV),ALLOWED,ALLOWED,ALLOWED
tulis file baru di raw,DENIED,DENIED,DENIED
lihat isi processed,ALLOWED,DENIED,ALLOWED
baca file Parquet processed,ALLOWED,DENIED,ALLOWED
tulis file baru di processed,ALLOWED,DENIED,DENIED
baca laporan quality,ALLOWED,DENIED,ALLOWED


Semua hasil sesuai rancangan hak akses.


In [4]:
print(subprocess.run(f'hdfs dfs -touchz {PROCESSED}/_acl_test', shell=True, capture_output=True, text=True,
                     env={**os.environ, 'HADOOP_USER_NAME': 'scientist'}).stderr.strip().splitlines()[-1])

touchz: Permission denied: user=scientist, access=WRITE, inode="/datalake/processed":axelnakata:engineer:drwxrwx---


## 3. Kelemahan autentikasi `simple`
`hadoop.security.authentication = simple` → HDFS **percaya begitu saja** pada nama user yang dikirim client. Siapa pun yang bisa menjangkau NameNode cukup mengaku sebagai `engineer`:

In [5]:
print('User OS sebenarnya :', sh('whoami', quiet=True))
print('Sebagai scientist  :', run_as('scientist', f'hdfs dfs -touchz {PROCESSED}/_spoof_test'))
print('Mengaku engineer   :', run_as('engineer',  f'hdfs dfs -touchz {PROCESSED}/_spoof_test'))
sh(f'hdfs dfs -ls {PROCESSED}/_spoof_test'); sh(f'hdfs dfs -rm -skipTrash {PROCESSED}/_spoof_test')

User OS sebenarnya : axelnakata


Sebagai scientist  : DENIED


Mengaku engineer   : ALLOWED


$ hdfs dfs -ls /datalake/processed/_spoof_test
-rw-rw----+  1 engineer engineer          0 2026-09-23 23:09 /datalake/processed/_spoof_test


$ hdfs dfs -rm -skipTrash /datalake/processed/_spoof_test
Deleted /datalake/processed/_spoof_test


'Deleted /datalake/processed/_spoof_test'

Orang yang sama ditolak sebagai `scientist`, tetapi langsung diizinkan hanya dengan mengubah 1 variabel lingkungan. Tidak ada password atau tiket. **Permission tanpa autentikasi kuat hanya melindungi dari kesalahan, bukan dari orang yang berniat jahat.** Solusi produksi: **Kerberos** (setiap user/servis harus punya tiket dari KDC) → rancangan Bab 3.1.

## 4. Audit log
Setiap akses ke NameNode sekarang tercatat: waktu, diizinkan atau tidak, user (`ugi`), IP, perintah (`cmd`), path (`src`). Ini bukti untuk investigasi insiden & compliance.

In [6]:
LOG = os.path.expanduser('~/opt/hadoop/logs/hdfs-audit.log')
pat = re.compile(r'^(\S+ \S+) INFO \S+: allowed=(\w+)\s+ugi=(\S+).*?ip=/(\S+)\s+cmd=(\S+)\s+src=(\S+)\s+dst=(\S+)')
recs = [m.groups() for l in open(LOG) if (m := pat.match(l))]
audit = pd.DataFrame(recs, columns=['time', 'allowed', 'ugi', 'ip', 'cmd', 'src', 'dst'])
print('baris audit log:', len(audit))
demo = audit[audit.ugi.str.startswith(('engineer', 'scientist', 'guest')) & audit.src.str.contains('_acl_test|_spoof_test|exotic_metadata')]
demo.tail(15)

baris audit log: 39883


,time,allowed,ugi,ip,cmd,src,dst
39831,"2026-09-24 06:09:43,745",true,guest,127.0.0.1,getfileinfo,/datalake/raw/exotic/_acl_test_guest,null
39833,"2026-09-24 06:09:43,766",false,guest,127.0.0.1,create,/datalake/raw/exotic/_acl_test_guest,null
39857,"2026-09-24 06:09:52,224",true,engineer,127.0.0.1,getfileinfo,/datalake/processed/_acl_test_engineer,null
39859,"2026-09-24 06:09:52,245",true,engineer,127.0.0.1,create,/datalake/processed/_acl_test_engineer,null
39860,"2026-09-24 06:09:52,880",true,engineer,127.0.0.1,getfileinfo,/datalake/processed/_acl_test_engineer,null
39861,"2026-09-24 06:09:52,909",true,engineer,127.0.0.1,delete,/datalake/processed/_acl_test_engineer,null
39862,"2026-09-24 06:09:53,489",true,scientist,127.0.0.1,getfileinfo,/datalake/processed/_acl_test_scientist,null
39864,"2026-09-24 06:09:53,508",false,scientist,127.0.0.1,create,/datalake/processed/_acl_test_scientist,null
39865,"2026-09-24 06:09:54,031",false,guest,127.0.0.1,getfileinfo,/datalake/processed/_acl_test_guest,null
39871,"2026-09-24 06:09:56,335",true,scientist,127.0.0.1,getfileinfo,/datalake/processed/_acl_test,null


In [7]:
denied = audit[audit.allowed == 'false']
print('Akses yang DITOLAK tercatat:'); display(denied[['time', 'ugi', 'cmd', 'src']].tail(10))
save_md(demo.tail(20), 'audit_log_sample', 'Potongan audit log NameNode (hdfs-audit.log)', 'allowed=false = akses ditolak permission.')

Akses yang DITOLAK tercatat:


,time,ugi,cmd,src
39830,"2026-09-24 06:09:43,130",scientist,create,/datalake/raw/exotic/_acl_test_scientist
39833,"2026-09-24 06:09:43,766",guest,create,/datalake/raw/exotic/_acl_test_guest
39838,"2026-09-24 06:09:45,493",guest,getfileinfo,/datalake/processed/metadata
39855,"2026-09-24 06:09:50,547",guest,getfileinfo,/datalake/processed/metadata/dataset=exotic
39856,"2026-09-24 06:09:51,385",guest,getfileinfo,/datalake/processed/metadata/dataset=exotic
39864,"2026-09-24 06:09:53,508",scientist,create,/datalake/processed/_acl_test_scientist
39865,"2026-09-24 06:09:54,031",guest,getfileinfo,/datalake/processed/_acl_test_guest
39870,"2026-09-24 06:09:55,787",guest,getfileinfo,/datalake/quality/metadata_quality.json
39873,"2026-09-24 06:09:56,355",scientist,create,/datalake/processed/_acl_test
39876,"2026-09-24 06:09:56,930",scientist,create,/datalake/processed/_spoof_test


'/Users/axelnakata/Documents/BigData-Earthquake/outputs/tables/audit_log_sample.md'

## 5. Lineage (Compliance, Bab 3.2)
Setiap file di raw zone punya catatan asal (URL), ukuran, SHA-256, dan waktu masuk. Pengecekan ulang metadata (24-09-2026) juga tercatat.

In [8]:
lin = pd.read_json(f'{ROOT}/outputs/hdfs/lineage.jsonl', lines=True)
display(lin[[c for c in ['event', 'dataset', 'file', 'bytes', 'sha256', 'ingested_at', 'checked_at', 'result'] if c in lin.columns]])
sh('hdfs dfs -ls /datalake/_lineage')

,event,dataset,file,bytes,sha256,ingested_at,checked_at,result
0,NaN,comcat,comcat_metadata.csv,5.002573e+07,8cc7fba99351b833fbb7b9668a9250263e2e8ba26feab2367a8a03c9627d2946,2026-09-22 22:19:53+00:00,NaT,NaN
1,NaN,noise,noise_metadata.csv,5.028650e+06,eb24b914c8820f6356699fdad55336d1f1f4f3bdc0b77a7b99f9640548b588e2,2026-09-22 22:19:56+00:00,NaT,NaN
2,NaN,exotic,exotic_metadata.csv,1.476207e+06,2ace01d8759e2affebfd22a46aeeae982517097f68e434e9e061dd72fdd74a68,2026-09-22 22:19:59+00:00,NaT,NaN
3,NaN,exotic,exotic_waveforms.hdf5,4.111581e+09,cbea8fcca7f4e8b4a81acf5747338f19d12de8be8f062368a4f83b77718614dd,2026-09-22 22:29:58+00:00,NaT,NaN
4,NaN,noise,noise_waveforms.hdf5,1.852757e+10,46c48dcf88402e1657556ae41833f1de3905b3bc100d3ec7134b8ac8877392b2,2026-09-22 23:18:51+00:00,NaT,NaN
5,NaN,comcat,comcat_waveforms.hdf5,6.734358e+10,27f9c82d7fd0329f7db9877069e5b27a90a0de24ab1c17694fea49b1d3901e38,2026-09-23 02:01:01+00:00,NaT,NaN
6,metadata_recheck,NaN,NaN,NaN,"{'comcat_metadata.csv': '8cc7fba99351b833fbb7b9668a9250263e2e8ba26feab2367a8a03c9627d2946', 'noise_metadata.csv': 'e...",NaT,2026-09-23 22:03:13+00:00,identical: UW-ESS == GDrive == raw zone copies (sha256); no new version stored


$ hdfs dfs -ls /datalake/_lineage
Found 1 items
-rw-r--r--   1 axelnakata supergroup       2992 2026-09-23 22:37 /datalake/_lineage/lineage.jsonl


'Found 1 items\n-rw-r--r--   1 axelnakata supergroup       2992 2026-09-23 22:37 /datalake/_lineage/lineage.jsonl'

## 6. Rancangan (tidak diimplementasikan di laptop)

| Aspek | Rancangan untuk produksi | Status di project ini |
|---|---|---|
| Autentikasi | **Kerberos** (KDC), tiap user & servis punya principal | `simple` → kelemahannya didemokan (bagian 3) |
| Otorisasi | Permission + ACL per zona per peran; **Apache Ranger** untuk kebijakan terpusat & per kolom | Permission + ACL ✅ (bagian 1–2) |
| Enkripsi saat disimpan | HDFS **encryption zone** + Hadoop KMS untuk raw & processed | tidak (disk laptop) |
| Enkripsi saat dikirim | TLS untuk Web UI/WebHDFS, `dfs.encrypt.data.transfer=true` | tidak |
| Audit | Audit log NameNode → dikirim ke SIEM, retensi mis. 1 tahun | audit log aktif ✅ (bagian 4) |
| Ketersediaan | `dfs.replication=3`, NameNode HA, `fs.trash.interval > 0` | replication 1, 1 NameNode, trash off; raw dilindungi 555 + salinan lokal + SHA-256 |
| Compliance | Lisensi: repo PNW-ML **MIT**, paper **CC BY 4.0**, data seismik PNSN/USGS publik → wajib atribusi (sitasi Ni dkk. 2023); lineage tiap file | lineage ✅ |
| Privacy | Data seismik **tidak berisi data pribadi**. Yang sensitif justru log akses (username, IP) → akses terbatas admin, retensi dibatasi. Koordinat stasiun = infrastruktur publik | klasifikasi data ✅ (rancangan) |
| Data quality | Aturan cleaning + laporan kualitas (completeness, validity, uniqueness, consistency, orphans) | ✅ notebook 03, `/datalake/quality/` |